# Blog Writing Evaluation - LangSmith Lab (20-Example Run)

**Domain:** AI-assisted blog writing  
**LangSmith project:** `blog_eval_langsmith`  
**What this notebook does:**
1. Traces a live blog-generation call with `@traceable`.
2. Creates a **20-example** LangSmith dataset (`blog-writing-eval-20`).
3. Runs an evaluation experiment with three LLM-as-judge evaluators.
4. Compares `gpt-4o-mini` vs `gpt-4o` side-by-side (A/B).
5. Summarises scores and draws a plain-English conclusion.
6. Demonstrates reproducibility at `temperature=0.7`.
7. **Compares 10-example vs 20-example results** to check whether conclusions hold at larger scale.

In [ ]:
%pip install langsmith langchain-openai openai python-dotenv -q

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()
os.environ['LANGSMITH_TRACING'] = 'true'
os.environ['LANGSMITH_ENDPOINT'] = 'https://eu.api.smith.langchain.com'
os.environ['LANGSMITH_PROJECT'] = 'blog_eval_langsmith'
print('Project:', os.environ['LANGSMITH_PROJECT'])
print('Endpoint:', os.environ['LANGSMITH_ENDPOINT'])
print('API key set:', bool(os.getenv('LANGSMITH_API_KEY') or os.getenv('LANGCHAIN_API_KEY')))
print('OpenAI key set:', bool(os.getenv('OPENAI_API_KEY')))

In [ ]:
import json
import re
from statistics import mean
from typing import Any, Dict, List, Optional
import pandas as pd
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langsmith import Client, traceable
from langsmith.evaluation import EvaluationResult
client = Client()
print('LangSmith client ready.')

## 1. Tracing

`@traceable` logs every call to LangSmith automatically.

In [ ]:
@traceable(name='generate-blog')
def generate_blog(inputs: Dict[str, Any], model_name: str = 'gpt-4o-mini') -> Dict[str, str]:
    prompt = ChatPromptTemplate.from_messages([
        ('system', 'You are an expert blog writer. Write a 600-800 word blog post that is clear, useful, and easy to scan.'),
        ('user', 'Topic: {topic}\
Brief: {brief}\
Write the blog post now.'),
    ])
    chain = prompt | ChatOpenAI(model=model_name, temperature=0.7)
    result = chain.invoke({'topic': inputs['topic'], 'brief': inputs['brief']})
    return {'blog_post': getattr(result, 'content', str(result)).strip()}

sample = generate_blog({
    'topic': 'How AI can help small businesses with marketing',
    'brief': 'Audience: non-technical small business owners. Tone: friendly. Goal: explain 3 practical AI marketing tips.',
})
print(sample['blog_post'][:800], '\
...')

## 2. Dataset

20 blog-writing examples for evaluation.

In [ ]:
DATASET_NAME = 'blog-writing-eval-20'
BLOG_EXAMPLES = [
    {'id': 'blog_001', 'inputs': {'topic': 'How AI can help small businesses with marketing', 'brief': 'Audience: non-technical. Tone: friendly. Goal: explain 3-4 practical AI marketing tips.'}, 'outputs': {'outline': ['Intro', 'Section 1: AI for ideas', 'Section 2: AI for emails', 'Section 3: AI for insights'], 'key_requirements': ['3 use cases', 'Simple language', 'Headings']}},
    {'id': 'blog_002', 'inputs': {'topic': 'Using AI to overcome writers block', 'brief': 'Audience: creators. Tone: honest. Goal: show how AI helps drafting.'}, 'outputs': {'outline': ['Intro', 'AI for brainstorming', 'AI for outlines', 'Keeping your voice'], 'key_requirements': ['AI does not replace voice', '2 concrete prompts', 'Supportive tone']}},
    {'id': 'blog_003', 'inputs': {'topic': 'Planning a month of content with AI', 'brief': 'Audience: business owners. Tone: practical. Goal: create 30-day content plan.'}, 'outputs': {'outline': ['Intro', 'Define themes', 'AI to generate ideas', 'Calendar creation'], 'key_requirements': ['Clear steps', 'Batch ideas', 'Review and edit']}},
    {'id': 'blog_004', 'inputs': {'topic': 'Repurposing blog post into multiple formats', 'brief': 'Audience: professionals. Tone: efficient. Goal: turn blog into social posts, emails, scripts.'}, 'outputs': {'outline': ['Intro', 'Pillar content', 'Social snippets', 'Emails', 'Scripts'], 'key_requirements': ['3+ formats', 'Aligned message', 'Time-saving']}},
    {'id': 'blog_005', 'inputs': {'topic': 'Common mistakes when using AI for content', 'brief': 'Audience: AI creators. Tone: constructive. Goal: highlight 3-5 mistakes.'}, 'outputs': {'outline': ['Intro', 'Copying without edit', 'Ignoring voice', 'Over-automating'], 'key_requirements': ['3+ mistakes', 'Solutions', 'Constructive tone']}},
    {'id': 'blog_006', 'inputs': {'topic': 'Using AI to research blog topics', 'brief': 'Audience: beginner bloggers. Tone: educational. Goal: gather ideas and questions.'}, 'outputs': {'outline': ['Intro', 'AI for angles', 'Organize notes', 'Verify sources'], 'key_requirements': ['Verify AI research', 'Example prompt', 'Beginner-friendly']}},
    {'id': 'blog_007', 'inputs': {'topic': 'Writing product launch posts with AI', 'brief': 'Audience: founders. Tone: persuasive. Goal: draft launch announcements.'}, 'outputs': {'outline': ['Intro', 'Value prop', 'Headlines', 'Body structure'], 'key_requirements': ['Value and pain points', 'Edit for brand voice', 'Persuasive not hyped']}},
    {'id': 'blog_008', 'inputs': {'topic': 'Building content calendar for local business', 'brief': 'Audience: shop owners. Tone: friendly. Goal: simple weekly/monthly calendar.'}, 'outputs': {'outline': ['Intro', 'Choose themes', 'AI proposals', 'Review'], 'key_requirements': ['Local business goals', 'Repeatable routine', 'Friendly tone']}},
    {'id': 'blog_009', 'inputs': {'topic': 'Creating evergreen tutorials with AI', 'brief': 'Audience: marketers. Tone: strategic. Goal: timeless tutorials.'}, 'outputs': {'outline': ['Intro', 'Durable topics', 'Step-by-step', 'Optimize'], 'key_requirements': ['Why evergreen lasts', 'Update reminder', 'Practical advice']}},
    {'id': 'blog_010', 'inputs': {'topic': 'Maintaining brand voice with AI content', 'brief': 'Audience: marketing teams. Tone: professional. Goal: brand consistency.'}, 'outputs': {'outline': ['Intro', 'Define voice', 'AI with constraints', 'Human review'], 'key_requirements': ['Brand guidelines', 'Human review', 'Professional']}},
    {'id': 'blog_011', 'inputs': {'topic': 'Writing SEO-friendly blog posts with AI', 'brief': 'Audience: bloggers new to SEO. Tone: practical. Goal: ranking posts.'}, 'outputs': {'outline': ['Intro', 'Keywords', 'Meta titles', 'Structure'], 'key_requirements': ['Plain terms', 'SEO prompt example', 'Structure']}},
    {'id': 'blog_012', 'inputs': {'topic': 'Using AI for compelling case studies', 'brief': 'Audience: B2B marketers. Tone: professional. Goal: speed up writing.'}, 'outputs': {'outline': ['Intro', 'Client story', 'PSR arc', 'Edit accuracy'], 'key_requirements': ['PSR structure', 'Client story first', 'Quotes and data']}},
    {'id': 'blog_013', 'inputs': {'topic': 'Creating FAQ pages that rank on Google', 'brief': 'Audience: e-commerce. Tone: practical. Goal: customer questions answered.'}, 'outputs': {'outline': ['Intro', 'Generate questions', 'Direct answers', 'Featured snippets'], 'key_requirements': ['FAQ connection', 'Real questions', 'Short structured']}},
    {'id': 'blog_014', 'inputs': {'topic': 'Writing email newsletters with AI', 'brief': 'Audience: founders/creators. Tone: friendly. Goal: people actually read.'}, 'outputs': {'outline': ['Intro', 'Subject lines', 'Body draft', 'CTA'], 'key_requirements': ['Subject/preview', 'Personal sound', 'CTA']}},
    {'id': 'blog_015', 'inputs': {'topic': 'Drafting thought leadership articles', 'brief': 'Audience: executives. Tone: authoritative. Goal: polish ideas.'}, 'outputs': {'outline': ['Intro', 'Own insight', 'Structure', 'Voice editing'], 'key_requirements': ['Human idea first', 'Structure', 'Personal voice']}},
    {'id': 'blog_016', 'inputs': {'topic': 'Using AI for product descriptions', 'brief': 'Audience: e-commerce. Tone: results-focused. Goal: descriptions that sell.'}, 'outputs': {'outline': ['Intro', 'Product details', 'Benefits', 'A/B test'], 'key_requirements': ['Features vs benefits', 'Prompt with details', 'A/B testing']}},
    {'id': 'blog_017', 'inputs': {'topic': 'Creating step-by-step how-to guides', 'brief': 'Audience: educators. Tone: instructional. Goal: easy-to-follow guides.'}, 'outputs': {'outline': ['Intro', 'Define scope', 'Numbered steps', 'Examples'], 'key_requirements': ['Define first', 'Numbered structure', 'Examples']}},
    {'id': 'blog_018', 'inputs': {'topic': 'Writing your brand origin story', 'brief': 'Audience: founders. Tone: narrative. Goal: compelling story.'}, 'outputs': {'outline': ['Intro', 'Raw facts', 'Narrative arc', 'Edit'], 'key_requirements': ['Founder material', 'Narrative arc', 'Authenticity']}},
    {'id': 'blog_019', 'inputs': {'topic': 'AI tools for non-native English speakers', 'brief': 'Audience: international pros. Tone: empowering. Goal: reduce barrier.'}, 'outputs': {'outline': ['Intro', 'Grammar fix', 'Expand ideas', 'Cultural voice'], 'key_requirements': ['Address barrier fear', 'Grammar/expansion', 'Cultural perspective']}},
    {'id': 'blog_020', 'inputs': {'topic': 'Writing community and culture posts', 'brief': 'Audience: managers. Tone: warm. Goal: culture posts.'}, 'outputs': {'outline': ['Intro', 'Spotlights', 'Recaps', 'Tone'], 'key_requirements': ['2+ content types', 'Avoid corporate', 'Specific details']}},
]
print(f'Defined {len(BLOG_EXAMPLES)} blog examples.')

In [ ]:
existing = [d.name for d in client.list_datasets()]
if DATASET_NAME not in existing:
    dataset = client.create_dataset(DATASET_NAME, description='20 blog examples')
    client.create_examples(
        inputs=[e['inputs'] for e in BLOG_EXAMPLES],
        outputs=[e['outputs'] for e in BLOG_EXAMPLES],
        dataset_id=dataset.id,
    )
    print(f'Created dataset {DATASET_NAME}')
else:
    dataset = next(d for d in client.list_datasets() if d.name == DATASET_NAME)
    print(f'Dataset {DATASET_NAME} exists')
print(f'Dataset id: {dataset.id}')

## 3. Run Evaluation

In [ ]:
def target_gpt4o_mini(inputs: dict) -> dict:
    return generate_blog(inputs, model_name='gpt-4o-mini')

In [ ]:
COVERAGE_PROMPT = (
    'You are an expert evaluating a blog post for content coverage. '
    'Score 0-100 based on how well the post addresses the topic and brief. '
    'Return only a single integer.'
)
STRUCTURE_PROMPT = (
    'You are an expert evaluating a blog post for structure and clarity. '
    'Score 0-100 based on logical flow, headings, and scannability. '
    'Return only a single integer.'
)
TONE_PROMPT = (
    'You are an expert evaluating a blog post for tone and audience fit. '
    'Score 0-100 based on how well the tone matches the stated audience and brief. '
    'Return only a single integer.'
)

def _make_evaluator(system_prompt: str, feedback_key: str, judge_model: str = 'gpt-4o-mini'):
    def evaluator(inputs: dict, outputs: dict, reference_outputs: dict) -> EvaluationResult:
        try:
            blog_post = outputs.get('blog_post', '')
            topic = inputs.get('topic', '')
            brief = inputs.get('brief', '')
            requirements = reference_outputs.get('key_requirements', [])
            user_msg = '\n'.join([
                f'Topic: {topic}',
                f'Brief: {brief}',
                f"Requirements: {', '.join(requirements)}",
                '',
                'Blog post:',
                blog_post,
                '',
                'Reply with a single integer score between 0 and 100, nothing else.',
            ])
            prompt = ChatPromptTemplate.from_messages([
                ('system', system_prompt),
                ('user', user_msg),
            ])
            llm = ChatOpenAI(model=judge_model, temperature=0)
            response = (prompt | llm).invoke({})
            text = getattr(response, 'content', str(response)).strip()
            match = re.search(r'\b(\d{1,3})\b', text)
            score = float(match.group(1)) if match else 0.0
            score = max(0.0, min(100.0, score))
        except Exception:
            score = 0.0
        return EvaluationResult(key=feedback_key, score=score, comment='LLM judge score')
    evaluator.__name__ = feedback_key
    return evaluator

def make_evaluators(judge_model: str = 'gpt-4o-mini') -> list:
    return [
        _make_evaluator(COVERAGE_PROMPT, 'coverage', judge_model),
        _make_evaluator(STRUCTURE_PROMPT, 'structure_clarity', judge_model),
        _make_evaluator(TONE_PROMPT, 'tone_match', judge_model),
    ]
print('Evaluators defined.')

In [ ]:
results_mini = client.evaluate(
    target_gpt4o_mini,
    data=DATASET_NAME,
    evaluators=make_evaluators(),
    experiment_prefix='gpt-4o-mini',
    max_concurrency=2,
)
print(results_mini)

## 4. Compare Models

In [ ]:
def target_gpt4o(inputs: dict) -> dict:
    return generate_blog(inputs, model_name='gpt-4o')

results_4o = client.evaluate(
    target_gpt4o,
    data=DATASET_NAME,
    evaluators=make_evaluators(),
    experiment_prefix='gpt-4o',
    max_concurrency=2,
)
print(results_4o)

## 5. Summary

In [ ]:
def extract_scores(results, feedback_keys=('coverage', 'structure_clarity', 'tone_match')):
    buckets = {k: [] for k in feedback_keys}
    rows = getattr(results, '_results', None) or getattr(results, 'results', None) or []
    for row in rows:
        for fb in (getattr(row, 'feedback', None) or []):
            key = getattr(fb, 'key', None)
            raw = getattr(fb, 'score', None)
            if key in buckets and raw is not None:
                buckets[key].append(float(raw))
    return {'n': len(rows), **{k: round(float(mean(v)), 1) if v else 0.0 for k, v in buckets.items()}}

scores_mini = extract_scores(results_mini)
scores_4o = extract_scores(results_4o)
summary = pd.DataFrame([{'model': 'gpt-4o-mini', **scores_mini}, {'model': 'gpt-4o', **scores_4o}])
print(summary.to_string(index=False))

In [ ]:
dims = ['coverage', 'structure_clarity', 'tone_match']
for row in summary.itertuples(index=False):
    scores = {d: getattr(row, d, 0.0) for d in dims}
    strongest = max(scores, key=scores.get)
    print(f"{row.model}: coverage={scores['coverage']:.0f} structure={scores['structure_clarity']:.0f} tone={scores['tone_match']:.0f} -> {strongest}")

## 6. Reproducibility

In [ ]:
repro_input = BLOG_EXAMPLES[0]['inputs']
run1 = generate_blog(repro_input)
run2 = generate_blog(repro_input)
print('Run 1 word count:', len(run1['blog_post'].split()))
print('Run 2 word count:', len(run2['blog_post'].split()))
print('Identical:', run1['blog_post'] == run2['blog_post'])
print('At temperature=0.7 outputs differ.')

## 7. Scale Check

In [ ]:
scores_10_mini = {'n': 10, 'coverage': 80.5, 'structure_clarity': 82.0, 'tone_match': 81.0}
scores_10_4o = {'n': 10, 'coverage': 78.5, 'structure_clarity': 83.0, 'tone_match': 82.0}
dims = ['coverage', 'structure_clarity', 'tone_match']
comparison = pd.DataFrame([
    {'run': '10-example', 'model': 'gpt-4o-mini', **{d: scores_10_mini[d] for d in dims}},
    {'run': '10-example', 'model': 'gpt-4o', **{d: scores_10_4o[d] for d in dims}},
    {'run': '20-example', 'model': 'gpt-4o-mini', **{d: scores_mini[d] for d in dims}},
    {'run': '20-example', 'model': 'gpt-4o', **{d: scores_4o[d] for d in dims}},
])
comparison['avg'] = comparison[dims].mean(axis=1).round(1)
print(comparison.to_string(index=False))

In [ ]:
for model in ['gpt-4o-mini', 'gpt-4o']:
    r10 = comparison[(comparison['run'] == '10-example') & (comparison['model'] == model)].iloc[0]
    r20 = comparison[(comparison['run'] == '20-example') & (comparison['model'] == model)].iloc[0]
    deltas = {d: round(r20[d] - r10[d], 1) for d in dims}
    avg_delta = round(r20['avg'] - r10['avg'], 1)
    print(f"{model}: coverage {deltas['coverage']:+.1f} structure {deltas['structure_clarity']:+.1f} tone {deltas['tone_match']:+.1f} avg={avg_delta:+.1f}")